# 01 — Basic Dynamics of Geomagnetic Toy Models

This notebook is a **fully fledged pedagogical introduction** to the basic logic of low-dimensional geomagnetic toy models.  
It is designed to fit the repository structure of `geomagnetic-toy-models` and to serve both as a teaching notebook and as a conceptual research notebook.

## Learning goals

By the end of this notebook, the reader should be able to:

- explain why geomagnetic toy models are useful,
- distinguish between deterministic drift and stochastic forcing,
- interpret a scalar dipole proxy as a reduced observable,
- compute elementary diagnostics such as polarity, residence times, and power spectra,
- connect simple time-series behavior to the broader repository structure.

## Repository integration

This notebook is written to be **compatible with the repo layout**, while remaining executable even when some repository scripts are still placeholders.  
The public repository snapshot currently exposes the notebook filenames and model/diagnostic folders, but several raw model scripts appear empty in the public view, so this notebook includes **self-contained canonical implementations** for teaching and experimentation.  


In [ ]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True

ROOT = Path.cwd()
if not (ROOT / "models").exists() and (ROOT.parent / "models").exists():
    ROOT = ROOT.parent

print("Working directory:", Path.cwd())
print("Repository root guessed as:", ROOT)

def repo_file_info(relpath):
    p = ROOT / relpath
    return {"exists": p.exists(), "size": p.stat().st_size if p.exists() else None, "path": str(p)}

def sign_changes(x):
    s = np.sign(x)
    s[s == 0] = np.nan
    valid = ~np.isnan(s)
    sv = s[valid]
    return int(np.sum(sv[1:] * sv[:-1] < 0))

def polarity_series(x):
    p = np.sign(x)
    if len(p) == 0:
        return p
    # carry last sign through zeros
    for i in range(1, len(p)):
        if p[i] == 0:
            p[i] = p[i-1]
    if p[0] == 0:
        nz = np.flatnonzero(p != 0)
        if len(nz):
            p[:nz[0]] = p[nz[0]]
    return p

def residence_times_from_signal(x, dt=1.0):
    p = polarity_series(np.asarray(x))
    if len(p) == 0:
        return np.array([])
    durations = []
    current = p[0]
    count = 1
    for val in p[1:]:
        if val == current:
            count += 1
        else:
            durations.append(count * dt)
            current = val
            count = 1
    durations.append(count * dt)
    return np.array(durations)

def power_spectrum(x, dt=1.0):
    x = np.asarray(x)
    x = x - np.mean(x)
    freqs = np.fft.rfftfreq(len(x), d=dt)
    spec = np.abs(np.fft.rfft(x))**2 / len(x)
    return freqs[1:], spec[1:]

rng = np.random.default_rng(42)


## 1. Quick repository check

The next cell inspects the expected repository files.  
If some model scripts are empty, the notebook still remains fully usable because the pedagogical implementations below are self-contained.


In [ ]:

files_to_check = [
    "models/bistable_models/double_well.py",
    "models/bistable_models/stochastic_forcing.py",
    "models/domino_model/simulate.py",
    "models/domino_model/analysis.py",
    "models/phase_oscillator_models/kuramoto_like.py",
    "models/data_driven_models/reduced_observables.py",
    "diagnostics/polarity.py",
    "diagnostics/reversal_statistics.py",
    "diagnostics/power_spectra.py",
]

for rel in files_to_check:
    info = repo_file_info(rel)
    print(f"{rel}: exists={info['exists']}, size={info['size']}")


## 2. The simplest reduced dipole model

A first conceptual model is a scalar observable \(x(t)\) representing a dipole-like quantity.  
The simplest linear stochastic dynamics is

$$
dx = -\lambda x\,dt + \sigma\,dW_t,
$$

where:

- \(\lambda > 0\) controls relaxation toward zero,
- \(\sigma\) controls unresolved fluctuations,
- \(W_t\) is a Wiener process.

This is not yet a reversal model. It is a **baseline fluctuating dynamics model**.


In [ ]:

def simulate_ou(n_steps=20000, dt=0.01, lam=0.4, sigma=0.6, seed=42):
    rng = np.random.default_rng(seed)
    x = np.zeros(n_steps)
    for i in range(1, n_steps):
        x[i] = x[i-1] + (-lam * x[i-1]) * dt + sigma * np.sqrt(dt) * rng.standard_normal()
    t = np.arange(n_steps) * dt
    return t, x

t, x = simulate_ou()
fig, ax = plt.subplots()
ax.plot(t, x, lw=1)
ax.set_title("Baseline fluctuating dipole proxy")
ax.set_xlabel("time")
ax.set_ylabel("x(t)")
plt.show()


## 3. From fluctuations to diagnostics

Even this simple model is useful for learning basic diagnostics.  
Below we compute:

- the polarity proxy,
- the number of sign changes,
- the residence-time distribution,
- the power spectrum.


In [ ]:

p = polarity_series(x)
res = residence_times_from_signal(x, dt=t[1]-t[0])
freqs, spec = power_spectrum(x, dt=t[1]-t[0])

print("Number of sign changes:", sign_changes(x))
print("Mean residence time:", res.mean())

fig, axes = plt.subplots(3, 1, figsize=(10, 10))
axes[0].plot(t, x, lw=1)
axes[0].set_title("Scalar dipole proxy")
axes[0].set_xlabel("time")
axes[0].set_ylabel("x")

axes[1].plot(t, p, lw=1)
axes[1].set_title("Polarity series")
axes[1].set_xlabel("time")
axes[1].set_ylabel("sgn(x)")

axes[2].loglog(freqs, spec)
axes[2].set_title("Power spectrum")
axes[2].set_xlabel("frequency")
axes[2].set_ylabel("power")

plt.tight_layout()
plt.show()


## 4. Parameter sensitivity

A research-grade notebook should not stop at a single run.  
We now explore how the variance of the process changes with \(\sigma\).


In [ ]:

sigmas = [0.2, 0.4, 0.8, 1.2]
fig, ax = plt.subplots()

for s in sigmas:
    _, xx = simulate_ou(sigma=s, seed=42)
    ax.plot(t, xx, label=f"sigma={s}", alpha=0.8)

ax.set_title("Sensitivity to noise amplitude")
ax.set_xlabel("time")
ax.set_ylabel("x(t)")
ax.legend()
plt.show()


## 5. Interpretation

This notebook should be understood as a **zeroth-order building block**:

- it introduces the language of reduced observables,
- it provides a first encounter with stochastic time series,
- it prepares the reader for the more physically meaningful bistable and domino-type models.

In the context of the repository, this notebook is best seen as a conceptual precursor to:

- `models/bistable_models/`,
- `models/domino_model/`,
- `diagnostics/polarity.py`,
- `diagnostics/power_spectra.py`.


## 6. Suggested exercises

1. Change `lam` and study how quickly the system forgets its previous state.  
2. Repeat the simulation with several random seeds and compare spectra.  
3. Add a moving-average smoother and compare the apparent residence times.  
4. Replace the linear drift with a cubic drift and compare with Notebook 02.
